# Agentic RAG Routing: Retrieve, Calculate, or Abstain

| Field | Value |
|---|---|
| Stage | LangGraph and agentic RAG |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Routing should choose the smallest authorized path that can answer the question, with an explicit abstention branch.

## 30-Second Summary

This notebook implements a deterministic three-route controller: policy questions retrieve, arithmetic questions use a calculator, and unsupported weather questions abstain. A three-case routing table verifies path and answer behavior.

## Why This Matters

Sending every question through retrieval or a general agent adds irrelevant context, cost, and risk. Explicit route criteria make the system predictable and testable.

## Scope

| Covers | Does not cover |
|---|---|
| Route contract, retrieval/tool/abstain branches, routing accuracy | LLM router, ambiguous multi-intent queries, external tools |


## Mental Model

```text
question -> classify route -> retrieve | calculate | abstain -> typed result
```


In [1]:
import re

POLICY = "Employees receive twenty days of paid leave each year."

def route(question: str) -> str:
    lowered = question.lower()
    if any(term in lowered for term in ("leave", "holiday", "policy")): return "retrieve"
    if re.fullmatch(r"\s*what is \d+ plus \d+\?\s*", lowered): return "calculate"
    return "abstain"


## How It Works

The router emits one of a closed set of labels. Each branch validates its own inputs and returns a common result shape containing route, answer, and optional citation.


## Baseline

A retrieve-everything baseline returns the leave policy even for arithmetic and weather questions, creating irrelevant context.


In [2]:
def retrieve_everything(question: str) -> dict:
    return {"route": "retrieve", "answer": POLICY, "citation": "leave"}

[retrieve_everything(question) for question in ("How much leave?", "What is 5 plus 8?", "Will it rain?")]


[{'route': 'retrieve',
  'answer': 'Employees receive twenty days of paid leave each year.',
  'citation': 'leave'},
 {'route': 'retrieve',
  'answer': 'Employees receive twenty days of paid leave each year.',
  'citation': 'leave'},
 {'route': 'retrieve',
  'answer': 'Employees receive twenty days of paid leave each year.',
  'citation': 'leave'}]

## Technique Implementation

The branch implementation is intentionally small. In production the router can be learned, but the label set, branch contracts, fallback, and evaluation remain explicit.


In [3]:
def run(question: str) -> dict:
    selected = route(question)
    if selected == "retrieve":
        return {"route": selected, "answer": POLICY, "citation": "leave"}
    if selected == "calculate":
        left, right = map(int, re.findall(r"\d+", question))
        return {"route": selected, "answer": str(left + right), "citation": None}
    return {"route": selected, "answer": "I cannot answer with the available paths.", "citation": None}

run("How much annual holiday do employees get?")


{'route': 'retrieve',
 'answer': 'Employees receive twenty days of paid leave each year.',
 'citation': 'leave'}

## Controlled Experiment

We evaluate one labeled example per route. Route accuracy and output checks are separate so a correct route with a broken branch cannot pass.


In [4]:
cases = [
    ("How much annual holiday do employees get?", "retrieve", "twenty"),
    ("What is 5 plus 8?", "calculate", "13"),
    ("Will it rain tomorrow?", "abstain", "cannot answer"),
]
results = [
    {"question": question, "expected_route": expected, "result": run(question), "answer_check": expected_text in run(question)["answer"].lower()}
    for question, expected, expected_text in cases
]
results


[{'question': 'How much annual holiday do employees get?',
  'expected_route': 'retrieve',
  'result': {'route': 'retrieve',
   'answer': 'Employees receive twenty days of paid leave each year.',
   'citation': 'leave'},
  'answer_check': True},
 {'question': 'What is 5 plus 8?',
  'expected_route': 'calculate',
  'result': {'route': 'calculate', 'answer': '13', 'citation': None},
  'answer_check': True},
 {'question': 'Will it rain tomorrow?',
  'expected_route': 'abstain',
  'result': {'route': 'abstain',
   'answer': 'I cannot answer with the available paths.',
   'citation': None},
  'answer_check': True}]

## Evaluation

All three examples select the expected route and pass branch-specific answer checks. This tiny table proves the controller contract, not general natural-language classification accuracy.


In [5]:
assert all(row["result"]["route"] == row["expected_route"] for row in results)
assert all(row["answer_check"] for row in results)
assert results[0]["result"]["citation"] == "leave"
assert results[1]["result"]["citation"] is None
print("Agentic routing checks passed.")


Agentic routing checks passed.


## Decision Guide

| Question | Route |
|---|---|
| Knowledge-base fact | Retrieve |
| Deterministic arithmetic | Calculator/tool |
| Unsupported/out of scope | Abstain |
| Ambiguous multi-intent | Clarify or bounded plan |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Wrong branch | Router labels too vague | Labeled confusion matrix |
| Correct branch, wrong output | Branch contract broken | Test branch separately |
| Everything becomes agentic | No simple-route preference | Cost/risk-aware routing policy |
| Unsupported route improvises | Missing fallback | Explicit abstention |


## Production Notes

### Observability
Log route, confidence/rule, branch latency/cost, fallback, citation, and outcome label.

### Safety and Guardrails
Authorize tools and sources after routing; classification is not permission.

### Latency and Cost
Route simple deterministic work away from expensive model loops.


## Practice

Add a compound policy-plus-arithmetic question and decide whether to clarify or invoke a two-step plan.

## Recall

Toggle - Recall: What is the router's output?
A closed, testable path label—not an answer.

Toggle - Recall: Why keep abstention?
It prevents unsupported questions from being forced through an irrelevant branch.

## Sources

- [LangGraph routing](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- Repository-owned synthetic routing cases

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the closed three-route controller | Add ambiguous and adversarial routing cases |
